# Backend Comparison: HF Transformers vs SpaCy

This notebook compares the two backends on various metrics.

In [ ]:
import time
import pandas as pd
from anonymization import (
    HFTransformersNER,
    SpacyNER,
    PipelineConfig,
    Document,
    ModelBackend,
)

In [ ]:
# Test dataset
test_cases = [
    ("person", "John Smith went to Paris."),
    ("org", "Google announced new AI models."),
    ("location", "The conference is in Berlin, Germany."),
    ("date", "Meeting scheduled for 2024-12-25."),
    ("email", "Contact us at support@example.com."),
    ("mixed", "Dr. Sarah Johnson from Microsoft Research in Seattle will present on 2024-06-15. Email: sarah.j@microsoft.com"),
    ("spanish", "María García vive en Madrid. Email: maria@empresa.es, Tel: +34 91 123 45 67."),
    ("financial", "Invoice INV-001 for Apple Inc. $50,000. IBAN: GB29NWBK60161331926819."),
]

texts = [t for _, t in test_cases]
labels = [l for l, _ in test_cases]

In [ ]:
# Initialize models
hf_config = PipelineConfig(model_backend=ModelBackend.HF_TRANSFORMERS, model_name="dslim/bert-base-NER")
spacy_config = PipelineConfig(model_backend=ModelBackend.SPACY, spacy_model="en_core_web_trf")

hf_ner = HFTransformersNER(hf_config)
spacy_ner = SpacyNER(spacy_config)

In [ ]:
# Benchmark function
def benchmark(model, name, texts, runs=3):
    times = []
    all_entities = []
    
    for _ in range(runs):
        start = time.perf_counter()
        entities = model.predict_batch(texts)
        elapsed = (time.perf_counter() - start) * 1000
        times.append(elapsed)
        all_entities = entities
    
    avg_time = sum(times) / len(times)
    total_entities = sum(len(e) for e in all_entities)
    
    return {
        "backend": name,
        "avg_time_ms": round(avg_time, 2),
        "total_entities": total_entities,
        "entities_per_text": round(total_entities / len(texts), 2),
        "times_ms": [round(t, 2) for t in times],
    }

In [ ]:
# Run benchmarks
hf_results = benchmark(hf_ner, "HF Transformers", texts)
spacy_results = benchmark(spacy_ner, "SpaCy", texts)

print(f"HF Transformers: {hf_results['avg_time_ms']}ms avg, {hf_results['total_entities']} entities")
print(f"SpaCy: {spacy_results['avg_time_ms']}ms avg, {spacy_results['total_entities']} entities")

In [ ]:
# Detailed entity comparison
comparison_data = []

for (label, text), hf_ents, spacy_ents in zip(test_cases, hf_ner.predict_batch(texts), spacy_ner.predict_batch(texts)):
    hf_labels = {e.label.value for e in hf_ents}
    spacy_labels = {e.label.value for e in spacy_ents}
    
    comparison_data.append({
        "test_case": label,
        "text": text[:60] + "..." if len(text) > 60 else text,
        "hf_count": len(hf_ents),
        "spacy_count": len(spacy_ents),
        "hf_types": ", ".join(sorted(hf_labels)) or "None",
        "spacy_types": ", ".join(sorted(spacy_labels)) or "None",
        "hf_entities": "; ".join(f"{e.text}({e.label.value})" for e in hf_ents),
        "spacy_entities": "; ".join(f"{e.text}({e.label.value})" for e in spacy_ents),
    })

df = pd.DataFrame(comparison_data)
print(df.to_string())

In [ ]:
# Test different HF models
hf_models = [
    "dslim/bert-base-NER",
    "dbmdz/bert-large-cased-finetuned-conll03-english",
    "Jean-Baptiste/roberta-large-ner-english",
]

for model_name in hf_models:
    try:
        config = PipelineConfig(model_backend=ModelBackend.HF_TRANSFORMERS, model_name=model_name)
        ner = HFTransformersNER(config)
        entities = ner.predict_batch(texts[:3])
        total = sum(len(e) for e in entities)
        print(f"{model_name}: {total} entities")
    except Exception as e:
        print(f"{model_name}: ERROR - {e}")

In [ ]:
# Test different SpaCy models
spacy_models = [
    "en_core_web_sm",
    "en_core_web_md",
    "en_core_web_lg",
    "en_core_web_trf",
]

for model_name in spacy_models:
    try:
        config = PipelineConfig(model_backend=ModelBackend.SPACY, spacy_model=model_name)
        ner = SpacyNER(config)
        entities = ner.predict_batch(texts[:3])
        total = sum(len(e) for e in entities)
        print(f"{model_name}: {total} entities")
    except Exception as e:
        print(f"{model_name}: ERROR - {e}")